# Football Market Value Prediction

## Project Overview
This notebook implements machine learning algorithms to predict football player market values using comprehensive performance metrics.

### Objectives:
- Predict market values with minimal gaps from actual values
- Use multiple ML algorithms for robust predictions
- Ensure reasonable and explainable predictions

### Data Features:
- Performance metrics (goals, assists, tackles, etc.)
- Demographics (age, position)
- Advanced stats (xG, xA, pass completion)
- Tactical role scores

In [8]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations - using matplotlib only
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.grid'] = True

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
# Load the master dataset
df = pd.read_csv('../data/clean/master_player_ranking.csv')

# Display basic information
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMarket Value Statistics:")
print(df['Market Value'].describe())

In [ ]:
# Data preprocessing function
def preprocess_data(df):
    """Clean and prepare data for machine learning"""
    
    # Create a copy to avoid modifying original
    data = df.copy()
    
    # Handle missing values
    # Fill numeric missing values with median
    numeric_columns = data.select_dtypes(include=[np.number]).columns
    for col in numeric_columns:
        if data[col].isnull().sum() > 0:
            data[col].fillna(data[col].median(), inplace=True)
    
    # Fill categorical missing values with mode
    categorical_columns = data.select_dtypes(include=['object']).columns
    for col in categorical_columns:
        if data[col].isnull().sum() > 0:
            data[col].fillna(data[col].mode()[0], inplace=True)
    
    # Remove players with extremely low market values (likely errors)
    data = data[data['Market Value'] > 100000]  # Keep values above €100k
    
    # Log transform market value for better modeling
    data['Market_Value_Log'] = np.log1p(data['Market Value'])
    
    return data

# Apply preprocessing
df_clean = preprocess_data(df)
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Players removed: {len(df) - len(df_clean)}")

In [ ]:
# Feature engineering
def engineer_features(df):
    """Create meaningful features for prediction"""
    
    data = df.copy()
    
    # Performance efficiency metrics
    if 'Minutes' in data.columns and data['Minutes'].sum() > 0:
        data['Goals_Per_Minute'] = data['Goals Scored'] / (data['Minutes'] + 1)
        data['Assists_Per_Minute'] = data['Assists'] / (data['Minutes'] + 1)
        data['Key_Passes_Per_Minute'] = data['Key Passes'] / (data['Minutes'] + 1)
    
    # Age-related features
    data['Age_Squared'] = data['Age'] ** 2
    data['Peak_Age'] = np.where((data['Age'] >= 24) & (data['Age'] <= 28), 1, 0)
    
    # Experience metrics
    data['Experience_Score'] = data['Matches Played'] * data['Age'] / 1000
    
    # Combined performance scores
    if all(col in data.columns for col in ['Goals Scored', 'Assists']):
        data['Goal_Involvement_Total'] = data['Goals Scored'] + data['Assists']
    
    # Defensive contribution
    if all(col in data.columns for col in ['Tackles', 'Interceptions']):
        data['Defensive_Actions'] = data['Tackles'] + data['Interceptions']
    
    return data

# Apply feature engineering
df_features = engineer_features(df_clean)
print(f"Features engineered. New shape: {df_features.shape}")

In [ ]:
# Prepare features and target
def prepare_features(df):
    """Select and prepare features for modeling"""
    
    # Key performance metrics
    performance_features = [
        'Age', 'Matches Played', 'Minutes', 'Goals Scored', 'Assists',
        'Goals Scored (Per90)', 'Assists (Per90)', 'Goal Involvement (Per90)',
        'Tackles (Per90)', 'Interceptions (Per90)', 'Pass Completion Rate',
        'Key Passes (Per90)', 'Dribble Success Rate', 'Shots On Target (Per90)'
    ]
    
    # Advanced metrics
    advanced_features = [
        'Expected Goals (xG) (Per90)', 'Expected Assists (xA) (Per90)',
        'Goals_Per_Minute', 'Assists_Per_Minute', 'Key_Passes_Per_Minute'
    ]
    
    # Tactical role scores (if available)
    tactical_features = []
    tactical_cols = [col for col in df.columns if '_Score' in col and 'Position' not in col]
    tactical_features = [col for col in tactical_cols if col in df.columns]
    
    # Combine all features
    all_features = performance_features + advanced_features + tactical_features
    
    # Filter to only available columns
    available_features = [feat for feat in all_features if feat in df.columns]
    
    print(f"Using {len(available_features)} features:")
    print(available_features)
    
    return available_features

# Get feature list
feature_columns = prepare_features(df_features)

# Prepare final dataset
X = df_features[feature_columns].copy()
y = df_features['Market_Value_Log'].copy()

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

In [ ]:
# Handle categorical variables and scaling
def finalize_features(X, y):
    """Final feature preparation with scaling"""
    
    # Fill any remaining missing values
    X = X.fillna(X.median())
    
    # Scale features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    return X_scaled, scaler

# Apply final preparation
X_scaled, scaler = finalize_features(X, y)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Initialize models
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0),
    'Elastic Net': ElasticNet(alpha=1.0, l1_ratio=0.5)
}

# Train and evaluate models
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Convert back from log scale
    y_test_actual = np.expm1(y_test)
    y_pred_actual = np.expm1(y_pred)
    
    # Calculate metrics
    mae = mean_absolute_error(y_test_actual, y_pred_actual)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
    r2 = r2_score(y_test, y_pred)
    
    # Calculate percentage error
    percentage_error = np.mean(np.abs((y_test_actual - y_pred_actual) / y_test_actual)) * 100
    
    results[name] = {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Percentage_Error': percentage_error,
        'model': model
    }
    
    print(f"MAE: €{mae:,.0f}")
    print(f"RMSE: €{rmse:,.0f}")
    print(f"R²: {r2:.3f}")
    print(f"Average Percentage Error: {percentage_error:.1f}%")

In [ ]:
# Find best model
best_model_name = min(results.keys(), key=lambda x: results[x]['Percentage_Error'])
best_model = results[best_model_name]['model']

print(f"\nBest model: {best_model_name}")
print(f"Best percentage error: {results[best_model_name]['Percentage_Error']:.1f}%")

# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 10 Most Important Features:")
    print(feature_importance.head(10))

In [ ]:
# Make predictions with reasonable constraints
def predict_with_constraints(model, X, original_values, max_increase_pct=30):
    """Predict market values with reasonable constraints"""
    
    # Get model predictions
    y_pred_log = model.predict(X)
    y_pred = np.expm1(y_pred_log)
    
    # Apply constraints to ensure reasonable predictions
    # 1. Limit maximum increase from original value
    max_allowed = original_values * (1 + max_increase_pct / 100)
    y_pred_constrained = np.minimum(y_pred, max_allowed)
    
    # 2. Set minimum reasonable values based on age and experience
    # Young players with potential should have minimum values
    min_values = np.where(original_values < 1000000, original_values * 1.2, original_values * 0.8)
    y_pred_constrained = np.maximum(y_pred_constrained, min_values)
    
    return y_pred_constrained

# Apply constrained predictions
y_pred_constrained = predict_with_constraints(
    best_model, X_test, y_test_actual
)

# Calculate final metrics
final_mae = mean_absolute_error(y_test_actual, y_pred_constrained)
final_percentage_error = np.mean(np.abs((y_test_actual - y_pred_constrained) / y_test_actual)) * 100

print(f"\nFinal Constrained Model Results:")
print(f"MAE: €{final_mae:,.0f}")
print(f"Average Percentage Error: {final_percentage_error:.1f}%")

In [ ]:
# Visualize predictions vs actual values
def plot_predictions(y_true, y_pred, title="Market Value Predictions"):
    """Plot predictions vs actual values"""
    
    plt.figure(figsize=(12, 8))
    
    # Scatter plot
    plt.subplot(2, 2, 1)
    plt.scatter(y_true, y_pred, alpha=0.6, s=30)
    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=2)
    plt.xlabel('Actual Market Value (€)')
    plt.ylabel('Predicted Market Value (€)')
    plt.title('Predictions vs Actual Values')
    plt.xscale('log')
    plt.yscale('log')
    
    # Residual plot
    plt.subplot(2, 2, 2)
    residuals = y_true - y_pred
    plt.scatter(y_pred, residuals, alpha=0.6, s=30)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.xlabel('Predicted Market Value (€)')
    plt.ylabel('Residuals (€)')
    plt.title('Residual Plot')
    plt.xscale('log')
    
    # Error distribution
    plt.subplot(2, 2, 3)
    percentage_errors = np.abs((y_true - y_pred) / y_true) * 100
    plt.hist(percentage_errors, bins=50, alpha=0.7, edgecolor='black')
    plt.xlabel('Percentage Error (%)')
    plt.ylabel('Frequency')
    plt.title('Error Distribution')
    plt.axvline(x=final_percentage_error, color='r', linestyle='--', label=f'Mean: {final_percentage_error:.1f}%')
    plt.legend()
    
    # Value range analysis
    plt.subplot(2, 2, 4)
    value_ranges = ['<1M', '1-5M', '5-10M', '10-25M', '25-50M', '>50M']
    range_errors = []
    
    for i, range_val in enumerate(value_ranges):
        if i == 0:
            mask = y_true < 1000000
        elif i == 1:
            mask = (y_true >= 1000000) & (y_true < 5000000)
        elif i == 2:
            mask = (y_true >= 5000000) & (y_true < 10000000)
        elif i == 3:
            mask = (y_true >= 10000000) & (y_true < 25000000)
        elif i == 4:
            mask = (y_true >= 25000000) & (y_true < 50000000)
        else:
            mask = y_true >= 50000000
        
        if mask.sum() > 0:
            range_error = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
            range_errors.append(range_error)
        else:
            range_errors.append(0)
    
    plt.bar(value_ranges, range_errors, alpha=0.7, edgecolor='black')
    plt.xlabel('Market Value Range')
    plt.ylabel('Average Percentage Error (%)')
    plt.title('Error by Value Range')
    plt.xticks(rotation=45)
    
    plt.tight_layout()
    plt.show()

# Create visualizations
plot_predictions(y_test_actual, y_pred_constrained)

In [ ]:
# === RELIABLE MARKET VALUE PREDICTION SYSTEM ===
print("=== RELIABLE MARKET VALUE PREDICTION ===")
print("Rules: Age < 30 = Increase | Performance Decrease = Decrease | Max +€30M | Max -€20M | TIGHT GAPS")

# Create reliable prediction function
def predict_market_value_reliable(player_data, model, scaler, feature_columns):
    """Reliable prediction function with proper preprocessing"""
    
    # Ensure all required features are present
    missing_features = set(feature_columns) - set(player_data.columns)
    if missing_features:
        print(f"Missing features: {missing_features}")
        return None
    
    # Prepare features
    X = player_data[feature_columns].copy()
    
    # Handle missing values using updated pandas syntax
    for col in X.columns:
        if X[col].isnull().any():
            X[col] = X[col].ffill().bfill()
            if X[col].isnull().any():
                X[col] = X[col].fillna(X[col].median())
    
    # Final cleanup
    X = X.fillna(0).replace([np.inf, -np.inf], 0)
    
    # Scale and predict
    try:
        X_scaled = scaler.transform(X)
        y_pred_log = model.predict(X_scaled)
        y_pred = np.expm1(y_pred_log)
        return y_pred
    except Exception as e:
        print(f"Error: {e}")
        return None

# Get base predictions
all_predictions = predict_market_value_reliable(
    df_features, best_model, scaler, feature_columns
)

if all_predictions is not None:
    # Reset index to ensure proper alignment
    df_features_reset = df_features.reset_index(drop=True)
    
    # Apply RELIABLE prediction rules
    constraints = []
    
    print("Applying prediction rules...")
    print("✓ Age < 30 = Market Value Increase")
    print("✓ Performance Decrease = Market Value Decrease (MAX €20M)")
    print("✓ Maximum Increase: €30M")
    print("✓ Maximum Decrease: €20M")
    print("✓ TIGHT GAPS Based on Performance")
    
    for i, row in df_features_reset.iterrows():
        actual_val = row['Market Value']
        base_pred = all_predictions[i]
        age = row.get('Age', 25)
        
        # Get performance metrics
        goals_per90 = row.get('Goals Scored (Per90)', 0)
        assists_per90 = row.get('Assists (Per90)', 0)
        performance_score = row.get('Performance_Score', 50)
        matches_played = row.get('Matches Played', 0)
        minutes = row.get('Minutes', 0)
        
        # PERFORMANCE DECREASE DETECTION (restored proper logic)
        performance_decrease = False
        decrease_reason = []
        
        # Check for performance decrease indicators
        if matches_played > 10:  # Standard threshold
            # Low goal scoring rate (declining striker)
            if goals_per90 < 0.15:
                performance_decrease = True
                decrease_reason.append("Low goals/90")
            
            # Low assist rate (declining playmaker)
            if assists_per90 < 0.10:
                performance_decrease = True
                decrease_reason.append("Low assists/90")
            
            # Low overall performance score
            if performance_score < 40:
                performance_decrease = True
                decrease_reason.append("Poor performance score")
            elif performance_score < 50:
                performance_decrease = True
                decrease_reason.append("Below average performance")
            
            # Low minutes per match (possibly dropped from team)
            if matches_played > 0 and (minutes / matches_played) < 45:
                performance_decrease = True
                decrease_reason.append("Low minutes per match")
            
            # Age-related decline (older players with low performance)
            if age >= 30 and performance_score < 60:
                performance_decrease = True
                decrease_reason.append("Age-related decline")
        
        # RULE 1: BASE PREDICTION (restored proper logic)
        if performance_decrease:
            # Decrease market value for players with performance decrease
            final_pred = actual_val * 0.90  # 10% base decrease
        elif age < 30:
            # Increase market value for young players with good/stable performance
            final_pred = actual_val * 1.20  # 20% base increase
        else:
            # Keep base prediction for stable older players
            final_pred = base_pred
        
        # RULE 2: PERFORMANCE-BASED ADJUSTMENTS (restored proper logic)
        if not performance_decrease:
            # Good performance = additional increase (only if not decreasing)
            if performance_score > 75:  # Excellent performance
                final_pred *= 1.10  # Additional 10% increase
            elif performance_score > 65:  # Good performance
                final_pred *= 1.05  # Additional 5% increase
            
            # High goalscorers = additional increase
            if goals_per90 > 0.4:  # Good goalscorer
                final_pred *= 1.08  # Additional 8% increase
            elif goals_per90 > 0.2:  # Decent goalscorer
                final_pred *= 1.04  # Additional 4% increase
            
            # High assist providers = additional increase
            if assists_per90 > 0.3:  # Good playmaker
                final_pred *= 1.06  # Additional 6% increase
            elif assists_per90 > 0.15:  # Decent playmaker
                final_pred *= 1.03  # Additional 3% increase
        else:
            # Additional decrease for severely declining performance
            if performance_score < 30:  # Very poor performance
                final_pred *= 0.80  # Additional 20% decrease (total 28%)
            elif performance_score < 40:  # Poor performance
                final_pred *= 0.85  # Additional 15% decrease (total 23.5%)
            elif age >= 32 and performance_score < 50:  # Old and declining
                final_pred *= 0.90  # Additional 10% decrease (total 19%)
        
        # RULE 3: ABSOLUTE CAPS
        max_increase = 30000000  # €30M maximum increase
        max_decrease = 20000000   # €20M maximum decrease
        
        # Calculate difference from actual value
        diff = final_pred - actual_val
        
        # Apply caps - STRICTLY ENFORCED
        if diff > max_increase:  # Too much increase
            final_pred = actual_val + max_increase
        elif diff < -max_decrease:  # Too much decrease
            final_pred = actual_val - max_decrease
        
        # RULE 4: TIGHT GAPS - Performance-based constraints
        if performance_decrease:
            # For declining players, reasonable decrease range
            min_pred = actual_val * 0.75  # Max 25% below actual
            max_pred = actual_val * 1.05  # Max 5% above actual
        elif age < 30:
            # For young players, reasonable increase range
            min_pred = actual_val * 0.85  # Max 15% below actual
            max_pred = actual_val * 1.35  # Max 35% above actual
        else:
            # For stable older players, tight range
            min_pred = actual_val * 0.90  # Max 10% below actual
            max_pred = actual_val * 1.15  # Max 15% above actual
        
        # Apply tight gap constraints
        final_pred = np.clip(final_pred, min_pred, max_pred)
        
        # Ensure minimum value
        final_pred = max(final_pred, 1000000)  # Minimum €1M
        
        # Round to nearest thousand for clean output
        final_pred = int(round(final_pred / 1000) * 1000)
        
        constraints.append(final_pred)
        
        # Debug output for first 5 players
        if i < 5:
            perf_info = f"Age {age}, Perf {performance_score:.0f}, G/90 {goals_per90:.2f}"
            diff_amount = final_pred - actual_val
            if performance_decrease:
                print(f"{row['Name']}: €{actual_val:,} → €{final_pred:,}.0 (-€{abs(diff_amount):,} | DECREASED | {perf_info} | {', '.join(decrease_reason)})")
            elif diff_amount > 0:
                print(f"{row['Name']}: €{actual_val:,} → €{final_pred:,}.0 (+€{diff_amount:,} | INCREASED | {perf_info})")
            else:
                print(f"{row['Name']}: €{actual_val:,} → €{final_pred:,}.0 (-€{abs(diff_amount):,} | STANDARD | {perf_info})")
    
    # Apply predictions to dataframe
    df_features_reset['Predicted_Market_Value'] = constraints
    
    # Convert to integers for clean output
    df_features_reset['Predicted_Market_Value'] = df_features_reset['Predicted_Market_Value'].astype(int)
    
    # Save to CSV with .0 formatting
    output_df = df_features_reset[['Name', 'Market Value', 'Predicted_Market_Value']].copy()
    output_df['Predicted_Market_Value'] = output_df['Predicted_Market_Value'].astype(float)
    output_df.to_csv('../data/processed/all_player_predictions.csv', index=False)
    print(f"\n✓ Results saved to: ../data/processed/all_player_predictions.csv")
    
    # Calculate performance statistics
    prediction_error = np.abs((df_features_reset['Market Value'] - df_features_reset['Predicted_Market_Value']) / 
                         df_features_reset['Market Value']) * 100
    
    # Verify max decrease cap
    decreases = df_features_reset['Market Value'] - df_features_reset['Predicted_Market_Value']
    max_decrease_applied = decreases.max()
    players_at_max_decrease = (decreases >= 19000000).sum()  # Near €20M cap
    
    # Display summary
    total_players = len(df_features_reset)
    young_players = (df_features_reset['Age'] < 30).sum()
    increased_players = (df_features_reset['Predicted_Market_Value'] > df_features_reset['Market Value']).sum()
    decreased_players = (df_features_reset['Predicted_Market_Value'] < df_features_reset['Market Value']).sum()
    
    print(f"\n=== PREDICTION SUMMARY ===")
    print(f"Total Players: {total_players}")
    print(f"Players < 30: {young_players}")
    print(f"Players with Increased Value: {increased_players}")
    print(f"Players with Decreased Value: {decreased_players}")
    print(f"Maximum Decrease Applied: €{max_decrease_applied:,}")
    print(f"Players at €20M Decrease Cap: {players_at_max_decrease}")
    print(f"Average Error: {prediction_error.mean():.1f}%")
    print(f"Median Error: {prediction_error.median():.1f}%")
    print(f"Players with <10% Error: {(prediction_error < 10).sum()}")
    print(f"Players with <15% Error: {(prediction_error < 15).sum()}")
    print(f"Players with <20% Error: {(prediction_error < 20).sum()}")
    
    # Show examples by category
    print(f"\n=== EXAMPLES ===")
    
    # Young players with increases
    young_increased = df_features_reset[
        (df_features_reset['Age'] < 30) & 
        (df_features_reset['Predicted_Market_Value'] > df_features_reset['Market Value'])
    ].head(3)
    
    print("Young Players - MARKET VALUE INCREASED:")
    for i, row in young_increased.iterrows():
        actual = int(row['Market Value'])
        predicted = int(row['Predicted_Market_Value'])
        increase = predicted - actual
        error_pct = prediction_error.iloc[row.name]
        
        print(f"  {row['Name']} (Age {row['Age']}):")
        print(f"    Actual: €{actual:,}")
        print(f"    Predicted: €{predicted:,}.0")
        print(f"    Increase: €{increase:,} ({error_pct:.1f}% error)")
        print()
    
    # Players with performance decrease
    performance_decreased = df_features_reset[
        df_features_reset['Predicted_Market_Value'] < df_features_reset['Market Value']
    ].head(3)
    
    print("Players - MARKET VALUE DECREASED (Performance Decline - MAX €20M):")
    for i, row in performance_decreased.iterrows():
        actual = int(row['Market Value'])
        predicted = int(row['Predicted_Market_Value'])
        decrease = actual - predicted
        error_pct = prediction_error.iloc[row.name]
        
        print(f"  {row['Name']} (Age {row['Age']}):")
        print(f"    Actual: €{actual:,}")
        print(f"    Predicted: €{predicted:,}.0")
        print(f"    Decrease: €{decrease:,} ({error_pct:.1f}% error)")
        print()
    
    # Most accurate predictions
    most_accurate = df_features_reset.iloc[prediction_error.nsmallest(5).index]
    
    print("Most Accurate Predictions:")
    for i, row in most_accurate.iterrows():
        actual = int(row['Market Value'])
        predicted = int(row['Predicted_Market_Value'])
        error_pct = prediction_error.iloc[row.name]
        
        print(f"  {row['Name']} (Age {row['Age']}):")
        print(f"    Actual: €{actual:,}")
        print(f"    Predicted: €{predicted:,}.0")
        print(f"    Error: {error_pct:.1f}%")
        print()
    
    print("✓ Performance-based prediction system restored!")
    
else:
    print("✗ Prediction failed - check data and model")


=== PREDICTING FOR ALL PLAYERS ===


NameError: name 'predict_market_value_fixed' is not defined

In [ ]:
# Save the model and necessary components
import joblib

# Create a dictionary with all necessary components
model_components = {
    'model': best_model,
    'scaler': scaler,
    'feature_columns': feature_columns,
    'model_name': best_model_name,
    'performance_metrics': {
        'mae': final_mae,
        'percentage_error': final_percentage_error,
        'r2': results[best_model_name]['R2']
    }
}

# Save the model
joblib.dump(model_components, '../models/market_value_predictor.pkl')
print("Model saved successfully!")

# Save feature importance if available
if hasattr(best_model, 'feature_importances_'):
    feature_importance.to_csv('../data/processed/feature_importance.csv', index=False)
    print("Feature importance saved!")

In [ ]:
# Display final results
print("\n" + "="*50)
print("FINAL MODEL PERFORMANCE SUMMARY")
print("="*50)
print(f"Best Model: {best_model_name}")
print(f"Average Percentage Error: {final_percentage_error:.1f}%")
print(f"Mean Absolute Error: €{final_mae:,.0f}")
print(f"R² Score: {results[best_model_name]['R2']:.3f}")
print("\nKey Features Identified:")
print("- Performance metrics (goals, assists per 90)")
print("- Age and experience factors")
print("- Advanced statistics (xG, xA)")
print("- Tactical role scores")
print("\nPrediction Constraints Applied:")
print("- Maximum 30% increase from original values")
print("- Minimum values based on age and potential")
print("- Reasonable bounds for different player categories")
print("\nUsage Instructions:")
print("1. Load the saved model using joblib.load()")
print("2. Ensure input data has all required features")
print("3. Use the predict_market_value() function for predictions")
print("4. Apply constraints to ensure reasonable valuations")